In [ ]:
import json
import logging
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple

import torch
import wandb
import weave
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-bnb-4bit"
MAX_SEQ_LENGTH = 4096
SYSTEM_PROMPT = ""

# logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


def validate_and_clean_msg(msg: Dict[str, Any]) -> Dict[str, str] | None:
    # strict structure verification
    if not isinstance(msg, dict) or "role" not in msg or "content" not in msg:
        return None
    role, content = str(msg["role"]).strip(), str(msg["content"]).strip()
    if not content:
        return None
    return {"role": role, "content": content}


def load_source_file(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        logger.warning(f"Source file missing: {path}")
        return []

    valid_convos = []
    with open(p, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                messages = obj.get("messages", [])
                if not isinstance(messages, list) or len(messages) < 2:
                    continue

                cleaned_messages = []
                for m in messages:
                    validated = validate_and_clean_msg(m)
                    if validated:
                        cleaned_messages.append(validated)

                if len(cleaned_messages) >= 2:
                    # Inject system prompt at index 0
                    cleaned_messages.insert(
                        0, {"role": "system", "content": SYSTEM_PROMPT}
                    )
                    valid_convos.append({"messages": cleaned_messages})
            except Exception as e:
                logger.warning(f"Skipping corrupt row {idx} in {path}: {e}")

    logger.info(f"Loaded {len(valid_convos)} valid conversations from {path}")
    return valid_convos


def prepare_split_datasets(
    paths: List[str], eval_ratio: float = 0.1
) -> Tuple[Dataset, Dataset]:
    # combines clean streams and runs a deterministic split
    random.seed(42)
    all_convos = []

    for path in paths:
        all_convos.extend(load_source_file(path))

    random.shuffle(all_convos)

    split_idx = int(len(all_convos) * (1 - eval_ratio))
    train_slice, eval_slice = all_convos[:split_idx], all_convos[split_idx:]

    random.shuffle(train_slice)
    return Dataset.from_list(train_slice), Dataset.from_list(eval_slice)


def main():
    try:
        # observability warmup
        wandb.init(project="gur-prime", name="sft-qwen2.5-twin-v4")
        weave.init("gur-prime")

        data_paths = [
            "data/whatsapp-sft.jsonl",
            "data/instagram-sft.jsonl",
            "data/synthetic-sft.jsonl",
        ]

        train_ds, eval_ds = prepare_split_datasets(data_paths, eval_ratio=0.1)
        logger.info(
            f"Final Dataset Struct -> Train Size: {len(train_ds)} | Eval Size: {len(eval_ds)}"
        )

        # initialize base model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
            dtype=torch.bfloat16,
        )

        # ensure input gradients flow to embeddings cleanly
        model.enable_input_require_grads()

        def format_chat(example):
            text = tokenizer.apply_chat_template(
                example["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            return {"text": text.strip()}

        train_ds = train_ds.map(format_chat).filter(lambda x: len(x["text"]) > 0)
        eval_ds = eval_ds.map(format_chat).filter(lambda x: len(x["text"]) > 0)

        # qlora config
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            lora_alpha=32,
            lora_dropout=0,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ],
            use_gradient_checkpointing="unsloth",
        )

        # Setup Standard SFT Trainer
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            packing=False,
            args=SFTConfig(
                output_dir="outputs/gur-prime-3",
                per_device_train_batch_size=2,
                gradient_accumulation_steps=4,
                warmup_ratio=0.1,
                num_train_epochs=1,
                learning_rate=2e-4,
                max_grad_norm=1.0,
                logging_steps=5,
                eval_strategy="steps",
                eval_steps=50,
                save_strategy="steps",
                save_steps=50,
                load_best_model_at_end=False,
                fp16=False,
                bf16=True,
                report_to="wandb",
                seed=42,
                dataset_kwargs={"add_special_tokens": False},  # <--- here
            ),
        )

        # Execute full-sequence training
        trainer.train()

        # Save Checkpoint Artifacts
        model.save_pretrained("outputs/gur-prime-final")
        tokenizer.save_pretrained("outputs/gur-prime-final")
        logger.info("Pipeline successful. Twin LoRA artifacts committed.")
        wandb.finish()

    except Exception as e:
        logger.exception("Critical fault in pipeline execution.")
        raise


if __name__ == "__main__":
    main()